In [ ]:
import pandas as pd
import yfinance as yfin
import matplotlib.pyplot as plt
import datetime as dt
import numpy as np
import statsmodels.api as sm

In [ ]:
Tickers = [
    'ADANIENT.NS', 'ADANIPORTS.NS', 'APOLLOHOSP.NS', 'ASIANPAINT.NS', 'AXISBANK.NS',
    'BAJAJ-AUTO.NS', 'BAJFINANCE.NS', 'BAJAJFINSV.NS', 'BEL.NS', 'BHARTIARTL.NS',
    'CIPLA.NS', 'COALINDIA.NS', 'DRREDDY.NS', 'EICHERMOT.NS', 'ETERNAL.NS',
    'GRASIM.NS', 'HCLTECH.NS', 'HDFCBANK.NS', 'HDFCLIFE.NS', 'HINDALCO.NS',
    'HINDUNILVR.NS', 'ICICIBANK.NS', 'INDIGO.NS', 'INFY.NS', 'ITC.NS',
    'JIOFIN.NS', 'JSWSTEEL.NS', 'KOTAKBANK.NS', 'LT.NS', 'M&M.NS',
    'MARUTI.NS', 'MAXHEALTH.NS', 'NESTLEIND.NS', 'NTPC.NS', 'ONGC.NS',
    'POWERGRID.NS', 'RELIANCE.NS', 'SBILIFE.NS', 'SHRIRAMFIN.NS', 'SBIN.NS',
    'SUNPHARMA.NS', 'TCS.NS', 'TATACONSUM.NS', 'TMPV.NS', 'TATASTEEL.NS',
    'TECHM.NS', 'TITAN.NS', 'TRENT.NS', 'ULTRACEMCO.NS', 'WIPRO.NS',
]

#Tickers = ['RELIANCE.NS', 'HDFCBANK.NS', 'TCS.NS', 'INFY.NS', 'ITC.NS', 'HEROMOTOCO.NS', 'COALINDIA.NS', 
#          'TATASTEEL.NS', 'BRITANNIA.NS', 'APOLLOHOSP.NS']


In [ ]:
start = dt.datetime.now().replace(year=dt.datetime.now().year - 5)
end   = dt.datetime.now()

# 1) Daily close prices (unadjusted close — market cap uses actual traded price, not total-return adjusted)
prices = yfin.download(Tickers, start=start, end=end, auto_adjust=False)['Close']

# 2) Historical shares outstanding (SEPARATE call per ticker) -> align to price dates
#    get_shares_full returns an irregular time series of share counts; ffill onto trading days.
shares = {}
book   = {}   # annual book equity (Stockholders Equity), columns = report dates
EQUITY_KEYS = ['Common Stock Equity', 'Stockholders Equity',
               'Total Stockholder Equity', 'Total Equity Gross Minority Interest']

In [ ]:
for t in Tickers:
    tk = yfin.Ticker(t)
    # --- shares outstanding history ---
    try:
        s = tk.get_shares_full(start=start, end=end)
        if s is not None and len(s):
            s = s[~s.index.duplicated(keep='last')].sort_index()
            s.index = s.index.tz_localize(None)            # drop tz so it aligns with price index
            shares[t] = s.reindex(prices.index, method='ffill')
        else:
            shares[t] = pd.Series(index=prices.index, dtype=float)
    except Exception as e:
        print(f"[shares] {t}: {e}")
        shares[t] = pd.Series(index=prices.index, dtype=float)

    # --- book equity from the annual balance sheet (~4 years, patchy) ---
    try:
        bs = tk.balance_sheet
        row = next((k for k in EQUITY_KEYS if k in bs.index), None)
        book[t] = bs.loc[row].dropna() if row is not None else pd.Series(dtype=float)
    except Exception as e:
        print(f"[book] {t}: {e}")
        book[t] = pd.Series(dtype=float)

shares = pd.DataFrame(shares).reindex(prices.index).ffill()

# 3) Daily market cap = close * shares outstanding
mktcap = prices * shares          # DataFrame: dates x tickers

In [ ]:
# 4) Point-in-time size & book-to-market at a chosen formation date
def point_in_time(formation_date, book_lag_months=6):
    """Size = market cap on the formation date.
       B/M   = most recent book equity available at least `book_lag_months` before
               the formation date (so only info known by then is used), / market cap."""
    fd = pd.Timestamp(formation_date)
    cutoff = fd - pd.DateOffset(months=book_lag_months)
    rows = []
    mc_asof = mktcap.loc[:fd].ffill().iloc[-1]          # last known market cap on/before fd
    for t in Tickers:
        be_series = book.get(t, pd.Series(dtype=float))
        be_known = be_series[be_series.index <= cutoff]  # lag: only reported-before-cutoff figures
        be = be_known.iloc[be_known.index.argmax()] if len(be_known) else np.nan
        mc = mc_asof.get(t, np.nan)
        rows.append({'stock': t, 'mktcap': mc, 'book_equity': be,
                     'book_to_market': (be / mc) if (mc and mc == mc) else np.nan})
    df = pd.DataFrame(rows).set_index('stock')
    df['size_rank'] = df['mktcap'].rank()               # small -> large
    df['value_rank'] = df['book_to_market'].rank()      # low B/M = growth, high B/M = value
    return df

# Example: form portfolios as of 30 June 2026
ff_inputs = point_in_time('2026-06-30')
print(ff_inputs.sort_values('mktcap', ascending=False).round(3))
print(f"\nUsable rows (both size & B/M): {ff_inputs[['mktcap','book_to_market']].dropna().shape[0]} of {len(Tickers)}")

In [ ]:
riskFreeDailyReturns = 6.5/252   # 6.5% annual, expressed as daily percent

dailyReturns = prices.pct_change(1, fill_method=None) * 100

dailyReturns = dailyReturns - riskFreeDailyReturns

print(f"Observations used: {len(dailyReturns)}")
dailyReturns

In [ ]:
NiftyTicker = ['^NSEI']
nifty = yfin.download(NiftyTicker, start = start, end = end)
nifty['Returns'] = nifty['Close'].squeeze().pct_change(1) * 100

niftyExcessReturns = nifty['Returns'] - riskFreeDailyReturns
niftyExcessReturns

In [ ]:
bigStocks = ff_inputs.loc[ff_inputs['size_rank'] >= 30.0].index
smallStocks = ff_inputs.loc[ff_inputs['size_rank'] <= 20.0].index

bigCols   = [s for s in bigStocks   if s in dailyReturns.columns]
smallCols = [s for s in smallStocks if s in dailyReturns.columns]

SMB = dailyReturns[smallCols].mean(axis=1) - dailyReturns[bigCols].mean(axis=1)  # Small Minus Big

valueStocks = ff_inputs.loc[ff_inputs['value_rank'] >= 35.0].index
growthStocks = ff_inputs.loc[ff_inputs['value_rank'] <= 15.0].index

valueCols = [s for s in valueStocks   if s in dailyReturns.columns]
growthCols = [s for s in growthStocks   if s in dailyReturns.columns]

HML = dailyReturns[valueCols].mean(axis=1) - dailyReturns[growthCols].mean(axis=1)

MKT = niftyExcessReturns

factors = pd.DataFrame({'MKT': MKT, 'SMB': SMB, 'HML': HML})
factors.dropna().shape

In [ ]:
rows = []

for stock in dailyReturns.columns:
    df = pd.concat([dailyReturns[stock].rename('y'), factors], axis=1).dropna()
    
    X = sm.add_constant(df[['MKT', 'SMB', 'HML']])
    model = sm.OLS(df['y'], X).fit()
    
    rows.append({
        'stock': stock,
        'alpha': model.params['const'],
        'beta_mkt': model.params['MKT'],
        'beta_smb': model.params['SMB'],
        'beta_hml': model.params['HML'],
        'r2': model.rsquared,
        'n': int(model.nobs),
    })
    
ffm = pd.DataFrame(rows).set_index('stock')
ffm.round(3)